# wandb-finish — ex1: close a wandb run at the end of train

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `wandb-finish`. Running the final beacon cell reports progress against the `Logging: wandb.finish` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: wandb.finish` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wandb-finish`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wandb-finish"
DD_SUBTOPIC = "Logging: wandb.finish"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `wandb.finish()` — quick refresher

`wandb.finish()` closes the current wandb run: flushes any pending metrics, uploads the final media + artifacts, and tears down the background sync thread. After it returns, the run is marked `finished` in the dashboard and a fresh `wandb.init(...)` will open a new one.

**Why you must call it explicitly.**

- Inside a Jupyter / Colab notebook the Python process keeps running after `train()` returns — wandb doesn't know training ended unless you tell it. The run sits in the `running` state on the dashboard forever.
- Inside a sweep, the agent calls `train()` repeatedly. If you don't finish each run, the second `wandb.init` either errors or silently appends to the wrong run.
- Exceptions skip `finish` unless you wrap in try/finally. ARENA's default loop puts it at the end of `train()` without a finally — that's fine for the drill but be aware.

**Symmetric pair.** Every `wandb.init(...)` (separate drill) wants a matching `wandb.finish()`.

### Exercise 1 — close a wandb run at the end of train

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `wandb.init` + `wandb.finish` as a symmetric pair around a fake training loop, verified by mocking the `wandb` module.
> Keywords: wandb, finish, lifecycle, mock
> ```

**KCs targeted:** `wandb-finish-after-train`, `wandb-init-finish-pair`

Implement `ex1_train_with_wandb_lifecycle(args, n_steps)`. A complete (mocked) train function that:

1. Opens a wandb run with `wandb.init(project=args.wandb_project, name=args.wandb_name)`. (No `config` kwarg this time — keep the signature minimal.)
2. Loops `for step in range(n_steps)` doing nothing inside (this drill is only about the lifecycle).
3. Calls `wandb.finish()` AFTER the loop.
4. Returns `n_steps`.

The test asserts the CALL ORDER: `wandb.init` first, then `wandb.finish`, exactly one of each.

In [ ]:
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def ex1_train_with_wandb_lifecycle(args, n_steps: int) -> int:
    """Open wandb run, fake-train n_steps, close run, return n_steps."""
    raise NotImplementedError()


def _test_ex1():
    from dataclasses import dataclass
    from unittest.mock import call

    @dataclass
    class FakeArgs:
        wandb_project: str = 'arena-transformer'
        wandb_name: str = 'lifecycle-test'

    # Build a parent mock so we can inspect call ORDER across init+finish.
    parent = MagicMock()
    parent.attach_mock(wandb.init, 'init')
    parent.attach_mock(wandb.finish, 'finish')
    wandb.init.reset_mock(); wandb.finish.reset_mock(); parent.reset_mock()
    parent.attach_mock(wandb.init, 'init')
    parent.attach_mock(wandb.finish, 'finish')

    args = FakeArgs()
    out = ex1_train_with_wandb_lifecycle(args, n_steps=5)
    assert out == 5, f'must return n_steps, got {out!r}'

    # init called exactly once, with project + name.
    assert wandb.init.call_count == 1, f'expected 1 wandb.init, got {wandb.init.call_count}'
    k = wandb.init.call_args.kwargs
    assert k.get('project') == 'arena-transformer'
    assert k.get('name') == 'lifecycle-test'
    # finish called exactly once.
    assert wandb.finish.call_count == 1, f'expected 1 wandb.finish, got {wandb.finish.call_count}'

    # Order: init BEFORE finish.
    names = [c[0] for c in parent.mock_calls]
    assert 'init' in names and 'finish' in names, f'missing calls: {names}'
    assert names.index('init') < names.index('finish'), (
        f'wandb.init must be called BEFORE wandb.finish, got order: {names}'
    )

    # Zero-step run still pairs init + finish.
    wandb.init.reset_mock(); wandb.finish.reset_mock()
    ex1_train_with_wandb_lifecycle(args, n_steps=0)
    assert wandb.init.call_count == 1
    assert wandb.finish.call_count == 1
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def ex1_train_with_wandb_lifecycle(args, n_steps: int) -> int:
    wandb.init(project=args.wandb_project, name=args.wandb_name)
    for step in range(n_steps):
        pass  # fake training step
    wandb.finish()
    return n_steps
```

**Why ORDER matters.** `wandb.finish()` flushes the buffer for the run opened by `wandb.init()`. Calling them out of order (or twice in a row) confuses the wandb sync thread. The test uses `parent.attach_mock` to record both calls into a single ordered list so we can assert `init` happened before `finish`.

**Zero-step edge case.** A run that does no training still needs the init/finish pair — wandb's run lifecycle is independent of what happens between. Skipping `finish` when `n_steps == 0` leaves a stale run on the dashboard.

**No try/finally here.** ARENA's default loop puts `finish` at the bare end of `train()` — an exception mid-loop will skip it. In production you'd wrap with try/finally; the drill keeps the ARENA pattern faithful.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()